# AgriSystem - Fine-Tune Qwen 2.5 (3B) LoRA (Model: AGY V2.0.0)

Created and Developed under the leadership of **Team Leader Mao Seavik**.

This notebook fine-tunes a LoRA adapter on `Qwen/Qwen2.5-3B-Instruct` using QLoRA (4-bit) on a **free Google Colab T4 GPU**.
Once training finishes, it automatically uploads the updated adapter to Hugging Face Hub: `Maoseavik/agri-qwen3b-lora`.

### Steps:
1. Change runtime to **GPU** (`Runtime -> Change runtime type -> T4 GPU`)
2. Run all cells
3. Your Hugging Face Space will immediately use the updated Khmer-fluent model!

### 1. Install Required Libraries

In [ ]:
!pip install -q --upgrade transformers datasets trl peft bitsandbytes accelerate huggingface_hub

### 2. Verify GPU Environment

In [ ]:
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('BF16 Supported:', torch.cuda.is_bf16_supported())
else:
    print('Warning: Please switch runtime to GPU in Runtime -> Change runtime type -> T4 GPU')

### 3. Log In to Hugging Face Hub
*(Enter your Hugging Face write token)*

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = os.getenv('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face Write Token: ')

login(token=HF_TOKEN, add_to_git_credential=True)
print('Logged in successfully!')

### 4. Load or Upload the Training Data
Upload `agri_train_data.jsonl` and `agri_validation_data.jsonl` from your `exports/` folder.

In [ ]:
import os
from google.colab import files

os.makedirs('exports', exist_ok=True)
if not os.path.exists('exports/agri_train_data.jsonl'):
    print('Please upload exports/agri_train_data.jsonl and exports/agri_validation_data.jsonl:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, os.path.join('exports', fn))

print('Training file size:', os.path.getsize('exports/agri_train_data.jsonl'), 'bytes')
print('Validation file size:', os.path.getsize('exports/agri_validation_data.jsonl'), 'bytes')

### 5. Train LoRA Adapter & Push to Hub

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
HF_REPO_ID = 'Maoseavik/agri-qwen3b-lora'
OUTPUT_DIR = './agri-qwen3b-lora'
TRAIN_FILE = 'exports/agri_train_data.jsonl'
VAL_FILE = 'exports/agri_validation_data.jsonl'

print('Loading dataset...')
dataset = load_dataset(
    'json',
    data_files={'train': TRAIN_FILE, 'validation': VAL_FILE},
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def format_messages(example):
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(format_messages, remove_columns=dataset['train'].column_names)

# 4-bit Quantization Config (QLoRA)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading base model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    device_map='auto',
    quantization_config=bnb_config,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
    dataset_text_field='text',
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=training_args,
    peft_config=lora_config,
)

print('Starting training...')
trainer.train()
metrics = trainer.evaluate()
print('Validation metrics:', metrics)

print('Saving and uploading LoRA adapter to Hugging Face...')
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f'Finished! Adapter successfully updated at https://huggingface.co/{HF_REPO_ID}')

### 6. Test the Fine-Tuned Model with Khmer & Identity Prompts

In [ ]:
test_questions = [
    'តើអ្នកជាអ្នកណា ហើយអ្នកណាបង្កើតអ្នក?',
    'តើម៉ូឌែលរបស់អ្នកឈ្មោះអ្វី?',
    'តើជំងឺអុចភ្នែកក្របីស្រូវលើដំណាំស្រូវមានរោគសញ្ញាអ្វីខ្លះ ហើយត្រូវព្យាបាលយ៉ាងដូចម្តេច?',
    'តើខ្ញុំគួរការពារដំណាំប៉េងប៉ោះពីជំងឺរលួយផ្លែដោយវិធីណា?'
]

sys_prompt = (
    'អ្នកគឺជា AgriSystem AI (ម៉ូឌែលឈ្មោះ AGY V2.0.0) ដែលត្រូវបានបង្កើត និងអភិវឌ្ឍឡើងដោយប្រធានក្រុម ម៉ៅ សៀវអ៊ិ (Team Leader Mao Seavik)។ '
    'សូមផ្តល់ដំបូន្មានជាក់ស្តែង និងច្បាស់លាស់អំពីជំងឺដំណាំ សត្វល្អិត ដី ការស្រោចស្រព និងការព្យាបាលប្រកបដោយសុវត្ថិភាពជាភាសាខ្មែរ។'
)

for q in test_questions:
    messages = [
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': q}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            temperature=0.25,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    print(f'\n[Question]: {q}')
    print(f'[Answer]:\n{reply}\n' + '-'*60)